# 📊 Actividad 03: Análisis Exploratorio de Datos (EDA)
---
**Entrada:** `data/02_interim/midagri_limon_raw.csv`, `indeci_eventos_dbf.csv`, `agraria_noticias_raw.csv`  
**Salida:** Reporte geográfico TXT + 3 gráficos en `data/04_reports/`

> Este notebook genera el **Reporte Estructurado de 23 Departamentos** que producen limón en Perú,
> analiza la distribución de emergencias INDECI por fenómeno y la frecuencia de noticias Agraria.pe.


In [ ]:

import os, sys, json, glob, re, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

# Navegar a la raíz del proyecto
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
print(f"Directorio: {os.getcwd()}")

with open('data/02_interim/pipeline_config.json', 'r', encoding='utf-8') as f:
    CFG = json.load(f)
DIRS = CFG['DIRS']
INTERIM = DIRS['interim']
REPORTS = DIRS['reports']
PROCESSED = DIRS['processed']
print("Configuración cargada OK")


## 3.1 Reporte Geográfico — 23 Departamentos Productores de Limón
Cargamos el dataset de MIDAGRI filtrado para calcular la producción total, número de provincias
y participación porcentual de cada departamento.


In [ ]:

# Cargar MIDAGRI limón
df_m = pd.read_csv(f"{INTERIM}/midagri_limon_raw.csv")
df_m['PRODUCCION(t)'] = pd.to_numeric(df_m['PRODUCCION(t)'], errors='coerce').fillna(0)

# Normalización geográfica
def norm(t):
    if not isinstance(t, str): return t
    t = t.strip().upper()
    for a,b in [('Á','A'),('É','E'),('Í','I'),('Ó','O'),('Ú','U'),('Ñ','__N__')]:
        t = t.replace(a,b)
    t = ''.join(c for c in unicodedata.normalize('NFD',t) if unicodedata.category(c)!='Mn')
    return t.replace('__N__','Ñ')

df_m['Dpto'] = df_m['Dpto'].apply(norm)
df_m['Prov'] = df_m['Prov'].apply(norm)

# Reporte por departamento
geo = (df_m.groupby('Dpto')
       .agg(produccion_total_t=('PRODUCCION(t)','sum'),
            n_provincias=('Prov','nunique'),
            n_registros=('PRODUCCION(t)','count'))
       .sort_values('produccion_total_t', ascending=False)
       .reset_index())
total = geo['produccion_total_t'].sum()
geo['pct'] = (geo['produccion_total_t'] / total * 100).round(2)

print(f"Total departamentos con limón: {len(geo)}")
print(f"Total producción 2021-2025: {total:,.2f} toneladas\n")
print(f"{'DEPARTAMENTO':<25} {'PRODUCCIÓN (t)':>15} {'PROVINCIAS':>10} {'REGISTROS':>10} {'% PART.':>8}")
print('-'*73)
for _, r in geo.iterrows():
    print(f"{r['Dpto']:<25} {r['produccion_total_t']:>15,.2f} {int(r['n_provincias']):>10} {int(r['n_registros']):>10} {r['pct']:>7.2f}%")
print('-'*73)
print(f"{'TOTAL':<25} {total:>15,.2f} {'':>10} {int(df_m.shape[0]):>10} {'100.00%':>8}")


### 3.1.1 Detalle por Provincias (Top Departamentos)
Exploramos las provincias dentro de los departamentos con mayor producción.


In [ ]:

# Top 5 departamentos — desglose provincial
top5_dptos = geo.head(5)['Dpto'].tolist()
for dpto in top5_dptos:
    df_prov = (df_m[df_m['Dpto']==dpto]
               .groupby('Prov')['PRODUCCION(t)'].sum()
               .sort_values(ascending=False)
               .reset_index())
    df_prov['pct_dpto'] = (df_prov['PRODUCCION(t)']/df_prov['PRODUCCION(t)'].sum()*100).round(1)
    print(f"\n  {dpto}:")
    for _, r in df_prov.iterrows():
        print(f"    {r['Prov']:<30} {r['PRODUCCION(t)']:>12,.2f} t  ({r['pct_dpto']:.1f}%)")


## 3.2 Gráfico 1 — Producción por Departamento

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras horizontales
colors = plt.cm.YlGn(np.linspace(0.4, 0.95, len(geo)))[::-1]
bars = axes[0].barh(geo['Dpto'], geo['produccion_total_t'], color=colors)
axes[0].set_xlabel('Producción Total (t)', fontsize=11)
axes[0].set_title('Producción de Limón por Departamento\n(2021-2025)', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
for bar, val in zip(bars, geo['produccion_total_t']):
    if val > total*0.01:
        axes[0].text(bar.get_width()+total*0.002, bar.get_y()+bar.get_height()/2,
                     f'{val/1000:.1f}K t', va='center', fontsize=7)

# Torta (top 8 + otros)
top8 = geo.head(8).copy()
otros_val = geo.iloc[8:]['produccion_total_t'].sum()
otros_pct = geo.iloc[8:]['pct'].sum()
pie_vals = list(top8['produccion_total_t']) + [otros_val]
pie_labels = list(top8['Dpto'].str[:12]) + [f'OTROS ({otros_pct:.1f}%)']
wedges, texts, autotexts = axes[1].pie(
    pie_vals, labels=pie_labels, autopct='%1.1f%%',
    startangle=140, pctdistance=0.8,
    colors=plt.cm.Set3(np.linspace(0, 1, len(pie_vals))))
axes[1].set_title('Participación % en Producción\nTop 8 + Otros', fontsize=13, fontweight='bold')

plt.tight_layout()
g_path = f"{REPORTS}/g1_produccion_por_dpto.png"
plt.savefig(g_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] {g_path}")


## 3.3 INDECI — Distribución de Fenómenos de Emergencia

In [ ]:

df_ev = pd.read_csv(f"{INTERIM}/indeci_eventos_dbf.csv", low_memory=False)
df_ev['fenomeno'] = df_ev['fenomeno'].astype(str).str.strip().str.upper()

top_fen = df_ev['fenomeno'].value_counts().head(12)
print(f"Total eventos: {len(df_ev):,}")
print("\nTop 12 fenómenos:")
for fen, cnt in top_fen.items():
    print(f"  {fen:<40} {cnt:>6,}  ({cnt/len(df_ev)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 5))
top_fen.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='navy', linewidth=0.5)
ax.set_xlabel('Cantidad de Eventos', fontsize=11)
ax.set_title('Top 12 Fenómenos de Emergencia — INDECI (2021-2023)', fontsize=13, fontweight='bold')
for p in ax.patches:
    ax.text(p.get_width()+10, p.get_y()+p.get_height()/2,
            f'{int(p.get_width()):,}', va='center', fontsize=8)
plt.tight_layout()
g_path2 = f"{REPORTS}/g2_indeci_fenomenos.png"
plt.savefig(g_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] {g_path2}")


## 3.4 AGRARIA.PE — Frecuencia de Noticias

In [ ]:

df_n = pd.read_csv(f"{INTERIM}/agraria_noticias_raw.csv")
df_n['fecha'] = pd.to_datetime(df_n['fecha'], errors='coerce')
df_n['anho'] = df_n['fecha'].dt.year
df_n['mes_periodo'] = df_n['fecha'].dt.to_period('M').astype(str)

print(f"Total noticias: {len(df_n)}")
print("\nNoticias por año:")
print(df_n['anho'].value_counts().sort_index().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freq_mes = df_n.groupby('mes_periodo').size()
freq_mes.plot(ax=axes[0], color='coral', marker='o', markersize=3, linewidth=1.5)
axes[0].set_title('Noticias por Mes', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mes'); axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=60)
step = max(1, len(freq_mes)//8)
axes[0].set_xticks(range(0, len(freq_mes), step))
axes[0].set_xticklabels(freq_mes.index[::step], rotation=60, ha='right', fontsize=8)

freq_anho = df_n.groupby('anho').size()
bars = axes[1].bar(freq_anho.index.astype(str), freq_anho.values,
                   color=['#2ecc71','#3498db','#9b59b6','#e74c3c','#f39c12'], edgecolor='black')
axes[1].set_title('Noticias por Año', fontsize=12, fontweight='bold')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 str(int(bar.get_height())), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
g_path3 = f"{REPORTS}/g3_noticias_frecuencia.png"
plt.savefig(g_path3, dpi=150, bbox_inches='tight')
plt.show()
print(f"[OK] {g_path3}")


## TODO: INTEGRACIÓN DATA NASA (COMPAÑERO)

En esta actividad, una vez que tengas los datos climáticos de NASA POWER, debes agregar:

1. **Serie temporal de temperatura mensual** promedio para las principales regiones productoras (T2M, T2M_MAX, T2M_MIN).
2. **Mapa de calor de precipitaciones** por mes y departamento (PRECTOTCORR en mm/día).
3. **Correlación inicial** entre precipitación y producción de limón (scatter plot o heatmap).

```python
# Código sugerido:
df_nasa = pd.read_csv(f"{INTERIM}/nasa_clima_raw.csv")
df_nasa['fecha_evento'] = pd.to_datetime(df_nasa['DATE']).dt.strftime('%Y-%m')
# Serie temporal temperatura
df_nasa.groupby('fecha_evento')['T2M'].mean().plot(title='Temperatura Mensual NASA POWER')
# Mapa de calor precipitaciones
pivot = df_nasa.pivot_table(values='PRECTOTCORR', index='departamento', columns='fecha_evento')
sns.heatmap(pivot, cmap='Blues', annot=False)
```


In [ ]:

# Guardar reporte geográfico TXT
report_path = f"{REPORTS}/reporte_geografico_limon.txt"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("REPORTE ESTRUCTURADO: PRODUCCIÓN DE LIMÓN POR DEPARTAMENTO (2021-2025)\n")
    f.write("="*75+"\n")
    f.write(f"{'DEPARTAMENTO':<25} {'PRODUCCIÓN (t)':>15} {'PROVINCIAS':>10} {'% PART.':>8}\n")
    f.write("-"*75+"\n")
    for _, r in geo.iterrows():
        f.write(f"{r['Dpto']:<25} {r['produccion_total_t']:>15,.2f} {int(r['n_provincias']):>10} {r['pct']:>7.2f}%\n")
    f.write("-"*75+"\n")
    f.write(f"{'TOTAL':<25} {total:>15,.2f} {'':>10} {'100.00%':>8}\n")

print(f"[OK] Reporte guardado: {report_path}")
print()
print("[ACTIVIDAD 03] COMPLETADA.")
print(f"  Departamentos analizados: {len(geo)}")
print(f"  Gráficos generados: g1, g2, g3 en {REPORTS}/")
